In [ ]:
# when did routines break, and whose: setup and load
import sys, time, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT = Path("/projects/ancz7294/yjmob")
PROC, FIG, CODE = ROOT / "data/processed", ROOT / "figures", ROOT / "code"
SEED = 20260917

C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "text.color": "#0b0b0b", "axes.labelcolor": "#52514e", "xtick.color": "#898781", "ytick.color": "#898781",
    "font.family": "sans-serif", "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight"})
def save(fig, name): fig.savefig(FIG / f"{name}.png"); plt.show(); plt.close(fig)

def table_png(df, name, title, width=11, mono=(), widths=None):
    # the table fills the whole figure, so the image has no empty band around it; every column gets the same 0.08 in of padding
    fig, ax = plt.subplots(figsize=(width, 0.3 * (len(df) + 1))); fig.subplots_adjust(0, 0, 1, 1); ax.axis("off"); ax.grid(False)
    w = np.asarray(widths if widths is not None else [1.0] * df.shape[1], float); w = w / w.sum()
    tb = ax.table(cellText=df.astype(str).values, colLabels=list(df.columns), cellLoc="left", colLoc="left", colWidths=list(w), bbox=[0, 0, 1, 1])
    tb.auto_set_font_size(False); tb.set_fontsize(8.5)
    for (r, c), cell in tb.get_celld().items():
        cell.set_edgecolor("#e1e0d9"); cell.PAD = 0.08 / (w[c] * width)
        if r == 0: cell.set_facecolor("#e8f0fb"); cell.set_text_props(weight="bold")
        elif df.columns[c] in mono: cell.set_text_props(family="monospace")
    ax.set_title(title, loc="left", fontweight="bold", fontsize=11, pad=8); save(fig, name)

t0 = time.time()
an = pd.read_parquet(PROC / "yjmob_analysis.parquet", columns=["uid", "d", "t", "x", "y"])
days = pd.read_parquet(PROC / "day_table.parquet").set_index("d").sort_index()
anc = pd.read_parquet(PROC / "person_anchors.parquet")
kinds = pd.read_parquet(PROC / "person_kinds.parquet")
ORDER = ["far roamers", "mid-range movers", "regular commuters", "local regulars", "weekend movers", "homebodies"]
COL = dict(zip(ORDER, [C[0], C[6], C[2], C[3], C[4], C[1]]))
rest_day = (~days.workday).to_numpy()                      # weekend-like and holiday-like days
uids = np.sort(kinds.index.to_numpy())                     # the 24,514 people with a kind of mover from Part 2
split = kinds.split.reindex(uids).to_numpy(); kind = kinds.kind.reindex(uids).to_numpy()
an = an[an.uid.isin(uids)]
print(f"loaded {len(an):,} rows for {len(uids):,} people with a kind in {time.time()-t0:.1f} s")
print(pd.Series(kind).value_counts().reindex(ORDER).to_string())

In [ ]:
%%writefile /projects/ancz7294/yjmob/code/breaks.py
"""When routines broke and whose broke, for 'Agent reliability under disruption', Part 3 (Naive Bayes and decision trees).
A person-day is 48 half-hour slots, each unseen or seen at home (3 x 3 block around the night anchor), at the daytime place
(3 x 3 block around the workday midday anchor) or elsewhere, with anchors from days 0-44 (profiles.py). Slot 0 (00:00) is left
out everywhere: it is thinly recorded on every emergency day, a recording artefact rather than behaviour.
"""
import numpy as np, pandas as pd
from scipy.special import gammaln

HOME, PLACE, ELSE = 1, 2, 3
STATES = ["home", "daytime place", "elsewhere"]
SLOTS = np.arange(1, 48)
MIN_SLOTS = 4

def slot_states(an, anc, uids):
    """S[person, day, slot]: 0 unseen, 1 home, 2 daytime place, 3 elsewhere, people in the order of uids."""
    a = an[["uid", "d", "t", "x", "y"]].join(anc[["home", "day_anchor"]], on="uid")
    near_home = np.maximum((a.x - a.home // 256).abs(), (a.y - a.home % 256).abs()) <= 1
    near_place = (np.maximum((a.x - a.day_anchor // 256).abs(), (a.y - a.day_anchor % 256).abs()) <= 1) & a.day_anchor.notna()
    st = np.where(near_home, HOME, np.where(near_place, PLACE, ELSE)).astype("int8")
    S = np.zeros((len(uids), 75, 48), dtype="int8")
    S[pd.Series(np.arange(len(uids)), index=uids).reindex(a.uid).to_numpy(), a.d.to_numpy(), a.t.to_numpy()] = st
    return S

def day_rows(S, uids, days, rest_day, min_slots=MIN_SLOTS):
    """Readable person-days (at least min_slots seen slots): row index i, uid, day, rest-day flag, seen slots."""
    n_seen = (S[:, :, SLOTS] > 0).sum(axis=2)
    i, d = np.nonzero(n_seen >= min_slots)
    keep = np.isin(d, days); i, d = i[keep], d[keep]
    return pd.DataFrame({"i": i, "uid": np.asarray(uids)[i], "d": d, "rest": rest_day[d], "seen": n_seen[i, d]})

def words(S, rows):
    """Multinomial NB input: each seen slot writes one word 'day type, slot, state' (2 x 47 x 3 = 282 possible words)."""
    k = S[rows.i.to_numpy(), rows.d.to_numpy()][:, SLOTS].astype("int16") - 1
    col = rows.rest.to_numpy()[:, None] * len(SLOTS) * 3 + np.arange(len(SLOTS))[None, :] * 3 + k
    W = np.zeros((len(rows), 2 * len(SLOTS) * 3), dtype="float32")
    r, c = np.nonzero(k >= 0); W[r, col[r, c]] = 1
    return W

def seen(S, rows, slots=SLOTS):
    """Bernoulli NB input: seen or not in each slot."""
    return (S[rows.i.to_numpy(), rows.d.to_numpy()][:, slots] > 0).astype("float32")

def counts(S, rows):
    """n[day type, slot, state] over the given person-days, seen slots only."""
    k = (S[rows.i.to_numpy(), rows.d.to_numpy()][:, SLOTS].astype("int16") - 1).ravel()
    rest = np.repeat(rows.rest.to_numpy().astype(int), len(SLOTS)); sl = np.tile(np.arange(len(SLOTS)), len(rows))
    n = np.zeros((2, len(SLOTS), 3)); m = k >= 0
    np.add.at(n, (rest[m], sl[m], k[m]), 1)
    return n

def person_counts(S, idx, days, rest_day):
    """n[person, day type, slot, state] for each person in idx over the given days, seen slots only."""
    n = np.zeros((len(idx), 2, len(SLOTS), 3))
    for d in days:
        k = S[idx, d][:, SLOTS].astype("int16") - 1
        r, s = np.nonzero(k >= 0)
        np.add.at(n, (r, int(rest_day[d]), s, k[r, s]), 1)
    return n

def smooth(n, a):
    """Laplace (additive) smoothing of categorical counts along the last axis."""
    return (n + a) / (n.sum(axis=-1, keepdims=True) + n.shape[-1] * a)

def shrink(n, prior, m):
    """Posterior mean under a Dirichlet(m * prior) prior: the person's counts pulled toward the prior by m pseudo-observations."""
    return (n + m * prior) / (n.sum(axis=-1, keepdims=True) + m)

def dm_loglik(n, prior, m):
    """Dirichlet-multinomial log marginal likelihood of counts n under Dirichlet(m * prior), summed over everything."""
    a = m * prior
    return float((gammaln(m) - gammaln(n.sum(axis=-1) + m) + (gammaln(n + a) - gammaln(a)).sum(axis=-1)).sum())

def best_m(n, prior, grid):
    """Empirical Bayes: the prior strength under which the observed counts are most likely."""
    ll = pd.Series({m: dm_loglik(n, prior, m) for m in grid})
    return ll.idxmax(), ll

def where_loglr(S, rows, theta_o, theta_e):
    """Evidence for 'emergency' over 'ordinary' (log-likelihood ratio, seen slots only) for each person-day. theta_* is either
    [day type, slot, state] (one model for everyone) or [person, day type, slot, state] with the person in rows.p."""
    k = S[rows.i.to_numpy(), rows.d.to_numpy()][:, SLOTS].astype("int16") - 1
    lr = np.log(theta_e) - np.log(theta_o)
    rest = rows.rest.to_numpy().astype(int); out = np.zeros(len(rows))
    p = rows.p.to_numpy() if lr.ndim == 4 else None
    for s in range(len(SLOTS)):
        m = k[:, s] >= 0
        out[m] += lr[rest[m], s, k[m, s]] if p is None else lr[p[m], rest[m], s, k[m, s]]
    return out

def surprise(S, rows, theta):
    """Total surprise (-log probability) and number of seen slots of each person-day under each person's habit model."""
    k = S[rows.i.to_numpy(), rows.d.to_numpy()][:, SLOTS].astype("int16") - 1
    rest = rows.rest.to_numpy().astype(int); p = rows.p.to_numpy()
    tot = np.zeros(len(rows)); cnt = np.zeros(len(rows))
    for s in range(len(SLOTS)):
        m = k[:, s] >= 0
        tot[m] -= np.log(theta[p[m], rest[m], s, k[m, s]]); cnt[m] += 1
    return tot, cnt

def cusum(L, k):
    """Page's CUSUM over the columns (days) of L: S = max(0, S + evidence - k)."""
    x = L.to_numpy() - k; s = np.zeros(len(x)); out = np.zeros_like(x)
    for j in range(x.shape[1]):
        s = np.maximum(0, s + x[:, j]); out[:, j] = s
    return pd.DataFrame(out, index=L.index, columns=L.columns)

def alarm_onset(C, h):
    """First day the CUSUM crosses h, and the onset: the day after it last stood at zero before the alarm."""
    hit = C.to_numpy() > h; cols = C.columns.to_numpy()
    alarm = pd.Series(np.nan, index=C.index); onset = pd.Series(np.nan, index=C.index)
    for r in np.where(hit.any(axis=1))[0]:
        f = hit[r].argmax(); z = np.where(C.iloc[r, :f + 1].to_numpy() == 0)[0]
        alarm.iloc[r] = cols[f]; onset.iloc[r] = cols[z[-1] + 1] if len(z) else cols[0]
    return alarm, onset

In [ ]:
# every person-day as 48 half hours: unseen, home, the daytime place or elsewhere (3 x 3 blocks around the Part 2 anchors)
sys.path.insert(0, str(CODE))
import importlib, breaks; importlib.reload(breaks)
from breaks import *
S = slot_states(an, anc, uids)
print(f"S: {S.shape[0]:,} people x {S.shape[1]} days x {S.shape[2]} half hours, {np.mean(S > 0):.3f} of them seen")
print("share of seen half hours at home, the daytime place, elsewhere:", pd.Series(S[S > 0]).value_counts(normalize=True).sort_index().round(3).tolist())
inside = np.maximum((anc.home // 256 - anc.day_anchor // 256).abs(), (anc.home % 256 - anc.day_anchor % 256).abs()) <= 1
print(f"people whose daytime place lies inside their home block (no separate daytime place): {inside.reindex(uids).mean():.1%}")

# how often each half hour is seen, ordinary fortnight against the emergency: the 00:00 slot is the odd one out
cov = pd.DataFrame({"ordinary": (S[:, 45:60] > 0).mean(axis=(0, 1)), "emergency": (S[:, 60:75] > 0).mean(axis=(0, 1))})
cov["ratio"] = cov.emergency / cov.ordinary
print(cov.ratio.describe().round(3).to_string()); print("lowest ratios:", cov.ratio.nsmallest(4).round(3).to_dict())
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(np.arange(48) / 2, cov.ordinary * 100, color=C[0], lw=2, label="days 45-59, ordinary")
axes[0].plot(np.arange(48) / 2, cov.emergency * 100, color=C[1], lw=2, label="days 60-74, emergency")
axes[0].set(xlabel="hour of day", ylabel="% of person-half-hours seen", xticks=range(0, 25, 3), title="How often people are seen"); axes[0].legend(frameon=False)
axes[1].bar(np.arange(48) / 2, cov.ratio, width=0.4, color=[C[7] if i == 0 else "#c3c2b7" for i in range(48)])
axes[1].axhline(cov.ratio[1:].median(), color="#52514e", lw=1, ls="--")
axes[1].set(xlabel="hour of day", ylabel="emergency / ordinary", xticks=range(0, 25, 3), ylim=(0.6, 1.0), title="00:00 is the odd one out")
save(fig, "f33_seen_by_half_hour")

In [ ]:
# labelled data: person-days from the ordinary fortnight (45-59, label 0) and the emergency (60-74, label 1), same mix of day types
rows = day_rows(S, uids, np.arange(45, 75), rest_day)
rows["y"] = (rows.d >= 60).astype(int); rows["split"] = split[rows.i]; rows["kind"] = kind[rows.i]; rows["p"] = rows.i
print(f"readable person-days (at least {MIN_SLOTS} seen half hours, 00:00 left out): {len(rows):,} of {len(uids) * 30:,}")
print("readable share by fortnight:", rows.groupby("y").size().div(len(uids) * 15).round(3).to_dict(), "| median seen half hours", rows.seen.median())
rng = np.random.default_rng(SEED)
dev_people = np.unique(rows.uid[rows.split == "dev"])
val_people = rng.choice(dev_people, len(dev_people) // 5, replace=False)          # for choosing the smoothing, inside development
train = rows[(rows.split == "dev") & ~rows.uid.isin(val_people)]; val = rows[rows.uid.isin(val_people)]; test = rows[rows.split == "eval"]
dev = rows[rows.split == "dev"]
print(f"train {len(train):,} days of {train.uid.nunique():,} people | validation {len(val):,} of {val.uid.nunique():,} | test {len(test):,} of {test.uid.nunique():,}")
print("people in more than one set:", len(set(train.uid) & set(test.uid)) + len(set(val.uid) & set(test.uid)) + len(set(train.uid) & set(val.uid)))

# images for the site: a day written as half-hour words, the counts the multinomial model reads, and the split
LETTER = np.array(["·", "H", "P", "E"])
ex = pd.concat([dev[dev.y == 0].sample(3, random_state=SEED), dev[dev.y == 1].sample(3, random_state=SEED)])
day_txt = ["".join(LETTER[S[i, d, SLOTS]]) for i, d in zip(ex.i, ex.d)]
t11 = pd.DataFrame({"uid": ex.uid.to_numpy(), "day": ex.d.to_numpy(), "day type": np.where(ex.rest, "rest", "work"),
                    "label": np.where(ex.y == 1, "emergency", "ordinary"), "half hours 00:30 to 23:30": day_txt})
table_png(t11, "t11_days_as_words", "Six labelled person-days, one letter per half hour\nH home, P daytime place, E elsewhere, · not seen", 7.4,
          mono=(t11.columns[-1],), widths=[0.09, 0.07, 0.11, 0.13, 0.60])
W = words(S, ex)
names = [f"{'rest' if r else 'work'} {(s // 2):02d}:{30 * (s % 2):02d} {st}" for r in (0, 1) for s in SLOTS for st in ("home", "place", "else")]
cols = [j for j in np.where(W.sum(axis=0) > 0)[0]][:9]
t12 = pd.DataFrame(W[:, cols].astype(int), columns=[names[j] for j in cols]); t12.insert(0, "label", t11.label); t12.insert(0, "uid", t11.uid)
table_png(t12.rename(columns=lambda c: c.replace("work ", "", 1)), "t12_word_counts", "The same days as word counts: 9 of the 282 words, here workday half hours (each seen half hour adds 1)", 11.5,
          widths=[0.07, 0.1] + [0.093] * 9)
t13 = pd.DataFrame({"set": ["training", "validation", "test"], "people": [f"{d_.uid.nunique():,}" for d_ in (train, val, test)],
                    "person-days": [f"{len(d_):,}" for d_ in (train, val, test)], "emergency share": [f"{d_.y.mean():.3f}" for d_ in (train, val, test)],
                    "who": ["development people, 80%", "development people, 20%", "evaluation people"]})
table_png(t13, "t13_nb_split", "Training, validation and test sets: split by person, nobody in two sets", 8, widths=[0.14, 0.12, 0.15, 0.18, 0.41])

In [ ]:
# three Naive Bayes models and their smoothing, chosen on the validation people
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score, confusion_matrix
def where_nb(fit_rows, a):
    """'Where' model from scratch: for each day type and half hour, the share of seen half hours at home, the place, elsewhere."""
    return smooth(counts(S, fit_rows[fit_rows.y == 0]), a), smooth(counts(S, fit_rows[fit_rows.y == 1]), a)
prior = np.log(train.y.mean() / (1 - train.y.mean()))
ALPHAS = [1e-4, 1e-3, 0.01, 0.1, 0.5, 1, 2, 5, 10]
MODELS = {"multinomial (where and when)": (words, MultinomialNB), "Bernoulli (when seen)": (seen, BernoulliNB)}
res = []
for nm, (f, cls) in MODELS.items():
    Xtr, Xva = f(S, train), f(S, val)
    for a in ALPHAS:
        p = cls(alpha=a, force_alpha=True).fit(Xtr, train.y).predict_proba(Xva)[:, 1]
        res.append((nm, a, accuracy_score(val.y, p > 0.5), log_loss(val.y, p)))
for a in ALPHAS:
    lr = where_loglr(S, val, *where_nb(train, a)) + prior; p = 1 / (1 + np.exp(-lr))
    res.append(("where only (from scratch)", a, accuracy_score(val.y, lr > 0), log_loss(val.y, np.clip(p, 1e-12, 1 - 1e-12))))
res = pd.DataFrame(res, columns=["model", "alpha", "accuracy", "log_loss"])
print(res.pivot(index="alpha", columns="model", values="accuracy").round(4).to_string())
print(res.pivot(index="alpha", columns="model", values="log_loss").round(4).to_string())
best_alpha = res.loc[res.groupby("model").log_loss.idxmin()].set_index("model").alpha
print("alpha with the lowest validation log loss:", best_alpha.to_dict())

# where smoothing really matters: one person's own days instead of thousands
own = person_counts(S, np.arange(len(uids)), np.arange(0, 45), rest_day)
print(f"a single person's 45 days leave {np.mean(own[(split == 'dev')] == 0):.1%} of their (day type, half hour, state) cells at zero;"
      f" with alpha = 0 any new day that lands in one of them would get probability 0")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for j, (nm, g) in enumerate(res.groupby("model", sort=False)):
    axes[0].plot(g.alpha, g.accuracy * 100, marker="o", lw=2, color=C[j], label=nm); axes[1].plot(g.alpha, g.log_loss, marker="o", lw=2, color=C[j])
axes[1].axhline(np.log(2), color="#52514e", lw=1, ls="--"); axes[1].text(1e-4, np.log(2), " a coin flip", va="bottom", fontsize=8.5, color="#52514e")
for ax, yl, t in [(axes[0], "validation accuracy (%)", "Accuracy hardly moves with alpha"), (axes[1], "validation log loss", "Log loss: how confident and how wrong")]:
    ax.set(xscale="log", xlabel="smoothing alpha", ylabel=yl, title=t)
axes[0].legend(frameon=False, fontsize=8.5)
save(fig, "f34_nb_smoothing")

In [ ]:
# final models on all development people, scored once on the evaluation people
out = {}
for nm, (f, cls) in MODELS.items():
    m = cls(alpha=best_alpha[nm], force_alpha=True).fit(f(S, dev), dev.y)
    out[nm] = m.predict_proba(f(S, test))[:, 1]
lr_where = where_loglr(S, test, *where_nb(dev, best_alpha["where only (from scratch)"])) + np.log(dev.y.mean() / (1 - dev.y.mean()))
out["where only (from scratch)"] = 1 / (1 + np.exp(-lr_where))
summary = pd.DataFrame({nm: {"accuracy": accuracy_score(test.y, p > 0.5), "AUC": roc_auc_score(test.y, p),
                             "log loss": log_loss(test.y, np.clip(p, 1e-12, 1 - 1e-12))} for nm, p in out.items()}).T
print(summary.round(4).to_string())
print(f"majority-class accuracy on the test days: {max(test.y.mean(), 1 - test.y.mean()):.4f}")
# checks on the Bernoulli result: is it a record counter, and what would the 00:00 artefact have added
print(f"number of seen half hours alone, AUC: {roc_auc_score(test.y, -test.seen):.4f}")
for nm, sl in [("all 48 half hours", np.arange(48)), ("00:00 alone", np.array([0]))]:
    p = BernoulliNB(alpha=best_alpha["Bernoulli (when seen)"]).fit(seen(S, dev, sl), dev.y).predict_proba(seen(S, test, sl))[:, 1]
    print(f"Bernoulli on {nm:18s} accuracy {accuracy_score(test.y, p > 0.5):.4f}, AUC {roc_auc_score(test.y, p):.4f}")
fig, axes = plt.subplots(1, 3, figsize=(13, 3.9))
for ax, (nm, p) in zip(axes, out.items()):
    cm = confusion_matrix(test.y, p > 0.5)
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max() * 1.3); ax.grid(False)
    for (r, c_), v in np.ndenumerate(cm): ax.text(c_, r, f"{v:,}\n{v / cm[r].sum():.0%} of row", ha="center", va="center", fontsize=9)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["ordinary", "emergency"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["ordinary", "emergency"])
    ax.set_xlabel("predicted"); ax.set_title(f"{nm}\naccuracy {accuracy_score(test.y, p > 0.5):.1%}", fontsize=10.5)
axes[0].set_ylabel("actual")
for ax in axes[1:]: ax.set_yticklabels([])
save(fig, "f35_nb_confusion")

In [ ]:
# empirical Bayes: one detector for everyone, one per kind of mover, one per person, each pulled toward the level above
a_w = best_alpha["where only (from scratch)"]
po, pe = where_nb(dev, a_w)
GRID_K = [1, 4, 16, 64, 256, 1024, 4096]
ko, ke, mk = {}, {}, {}
for k in ORDER:
    no, ne = counts(S, dev[(dev.kind == k) & (dev.y == 0)]), counts(S, dev[(dev.kind == k) & (dev.y == 1)])
    mk[k] = (best_m(no, po, GRID_K)[0], best_m(ne, pe, GRID_K)[0])
    ko[k], ke[k] = shrink(no, po, mk[k][0]), shrink(ne, pe, mk[k][1])
print("prior strength toward everyone (ordinary, emergency), in pseudo-observations per half hour:", mk)
# a person's habits from their own days 0-44, pulled toward their kind's days 0-44; the data choose how hard
kind_base = {k: smooth(own[(kind == k) & (split == "dev")].sum(axis=0), a_w) for k in ORDER}
prior_p = np.stack([kind_base[k] for k in kind])
GRID_P = [0.25, 0.5, 1, 2, 4, 8, 16, 32, 64]
m_p, ll_p = best_m(own[split == "dev"], prior_p[split == "dev"], GRID_P)
print(f"person-level prior strength: {m_p} (log marginal likelihood by strength: {ll_p.round(0).astype(int).to_dict()})")
base = shrink(own, prior_p, m_p)
tilt = np.stack([ke[k] / ko[k] for k in kind])                        # how each kind's day shifts in the emergency
def emer_of(b): e = b * tilt; return e / e.sum(axis=-1, keepdims=True)
lr_kind = np.zeros(len(test))
for k in ORDER:
    m = (test.kind == k).to_numpy(); lr_kind[m] = where_loglr(S, test[m], ko[k], ke[k])
lr_pers = where_loglr(S, test, base, emer_of(base))
pr0 = np.log(dev.y.mean() / (1 - dev.y.mean()))
levels = {"one for everyone": lr_where - pr0, "one per kind": lr_kind, "one per person": lr_pers}
rows_eb = []
for nm, lr in levels.items():
    lr = lr + pr0
    rows_eb.append((nm, "all", accuracy_score(test.y, lr > 0), roc_auc_score(test.y, lr), np.mean(lr[test.y.to_numpy() == 0] > 0)))
    for k in ORDER:
        m = (test.kind == k).to_numpy()
        rows_eb.append((nm, k, accuracy_score(test.y[m], lr[m] > 0), roc_auc_score(test.y[m], lr[m]), np.mean(lr[m & (test.y.to_numpy() == 0)] > 0)))
eb_tab = pd.DataFrame(rows_eb, columns=["detector", "kind", "accuracy", "AUC", "false_alarm"])
print(eb_tab.pivot(index="kind", columns="detector", values="false_alarm").reindex(["all"] + ORDER).round(3).to_string())
print(eb_tab.pivot(index="kind", columns="detector", values="AUC").reindex(["all"] + ORDER).round(3).to_string())
print(eb_tab.pivot(index="kind", columns="detector", values="accuracy").reindex(["all"] + ORDER).round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4), gridspec_kw={"width_ratios": [1, 1.6]})
axes[0].plot(ll_p.index, (ll_p - ll_p.max()) / 1e3, marker="o", color=C[0], lw=2); axes[0].axvline(m_p, color=C[1], ls="--", lw=1)
axes[0].set(xscale="log", xlabel="prior strength m (pseudo-observations per half hour)", ylabel="log marginal likelihood, thousands below best",
            title=f"The data choose m = {m_p}")
fa = eb_tab[eb_tab.kind != "all"].pivot(index="kind", columns="detector", values="false_alarm").reindex(ORDER)
xx = np.arange(len(ORDER)); w = 0.26
for j, nm in enumerate(levels):
    axes[1].bar(xx + (j - 1) * w, fa[nm] * 100, width=w, color=[C[7], C[3], C[2]][j], label=nm)
axes[1].axhline(50, color="#52514e", lw=1, ls="--"); axes[1].set_xticks(xx); axes[1].set_xticklabels([o.replace(" ", "\n", 1) for o in ORDER], fontsize=8.5)
axes[1].set(ylabel="% of ordinary days called emergency", title="False alarms on ordinary days, by kind"); axes[1].legend(frameon=False, fontsize=8.5); axes[1].xaxis.grid(False)
save(fig, "f36_empirical_bayes")

In [ ]:
# each person's day-by-day curve, days 0-74, evaluation people. For a day in days 0-44 the person's habits come from the other
# two fortnights of 0-44 (so no day judges itself); from day 45 on, from all of 0-44
ev_rows = day_rows(S, uids, np.arange(75), rest_day); ev_rows = ev_rows[split[ev_rows.i] == "eval"].reset_index(drop=True); ev_rows["p"] = ev_rows.i
BL = [np.arange(0, 15), np.arange(15, 30), np.arange(30, 45)]
own_b = [person_counts(S, np.arange(len(uids)), b, rest_day) for b in BL]
ev_rows["llr"] = np.nan
m = ev_rows.d >= 45; ev_rows.loc[m, "llr"] = where_loglr(S, ev_rows[m], base, emer_of(base))
for j, b in enumerate(BL):
    bj = shrink(sum(own_b[q] for q in range(3) if q != j), prior_p, m_p); m = ev_rows.d.isin(b)
    ev_rows.loc[m, "llr"] = where_loglr(S, ev_rows[m], bj, emer_of(bj))
L = ev_rows.pivot(index="uid", columns="d", values="llr").reindex(columns=range(75))
L = L.sub(L.loc[:, :44].mean(axis=1), axis=0)          # centred on the person's own ordinary days
post = 1 / (1 + np.exp(-L))
ek = pd.Series(kind, index=uids).reindex(L.index)
curve = post.groupby(ek).mean().reindex(ORDER)
per = {p_: (a_, b_) for p_, a_, b_ in [("0-44", 0, 44), ("45-59", 45, 59), ("60-74", 60, 74)]}
print("mean probability that a day is an emergency day, by period:")
print(pd.DataFrame({p_: curve.loc[:, a_:b_].mean(axis=1) for p_, (a_, b_) in per.items()}).round(3).to_string())
print("all evaluation people, days 57-63:", post.mean().loc[57:63].round(3).to_dict())

wd = days.index[days.workday].to_numpy()
print("workdays only, by period:", {p_: round(float(post.loc[:, [d for d in wd if a_ <= d <= b_]].mean().mean()), 3) for p_, (a_, b_) in per.items()},
      "| rest days only:", {p_: round(float(post.loc[:, [d for d in range(a_, b_ + 1) if d not in wd]].mean().mean()), 3) for p_, (a_, b_) in per.items()})
fig, ax = plt.subplots(figsize=(12, 4.3))
ax.axvspan(59.5, 74.5, color=C[1], alpha=0.07, lw=0); ax.text(60, 0.015, "emergency", color=C[1], fontsize=9, transform=ax.get_xaxis_transform())
for k in ORDER: ax.plot(wd, curve.loc[k, wd] * 100, color=COL[k], lw=1.6, marker="o", ms=2.5, label=k)
ax.plot(wd, post.mean()[wd] * 100, color="#0b0b0b", lw=2.6, label="everyone")
ax.axhline(50, color="#52514e", lw=0.8, ls="--")
ax.set(xlabel="day (workdays only)", ylabel="mean probability of 'emergency' (%)", xlim=(0, 74), title="When routines broke: each person's workdays read against their own habits")
ax.legend(frameon=False, fontsize=8.5, ncol=4, loc="upper left")
save(fig, "f37_break_curve")

In [ ]:
# who broke: how much more each person's habit model is surprised by the emergency fortnight than by the ordinary one, against
# the same rise between two ordinary fortnights (habits from 0-29, scored on 30-44 and 45-59: same gaps, no emergency)
idx = np.arange(len(uids))
def habits(fit_days):
    n = person_counts(S, idx, fit_days, rest_day)
    pr = np.stack([{k: smooth(n[(kind == k) & (split == "dev")].sum(axis=0), a_w) for k in ORDER}[k] for k in kind])
    return shrink(n, pr, best_m(n[split == "dev"], pr[split == "dev"], GRID_P)[0])
def fortnight(theta, lo, hi):
    r = day_rows(S, uids, np.arange(lo, hi + 1), rest_day, min_slots=1); r["p"] = r.i
    tot, cnt = surprise(S, r, theta)
    g = pd.DataFrame({"i": r.i, "tot": tot, "cnt": cnt}).groupby("i").sum()
    return (g.tot / g.cnt).reindex(idx).to_numpy(), g.cnt.reindex(idx).fillna(0).to_numpy()
h44, h29 = base, habits(np.arange(30))
lab = pd.DataFrame({"split": split, "kind": kind}, index=pd.Index(uids, name="uid"))
lab["S_ord"], n_ord = fortnight(h44, 45, 59); lab["S_emg"], n_emg = fortnight(h44, 60, 74)
lab["P_early"], n_pe = fortnight(h29, 30, 44); lab["P_late"], n_pl = fortnight(h29, 45, 59)
lab["delta"] = lab.S_emg - lab.S_ord; lab["delta_placebo"] = lab.P_late - lab.P_early
lab["seen_min"] = np.minimum(n_ord, n_emg); lab["seen_min_placebo"] = np.minimum(n_pe, n_pl)
MIN = 30
lab = lab[(lab.seen_min >= MIN) & (lab.seen_min_placebo >= MIN)].copy()
print(f"people with at least {MIN} seen half hours in each of the four fortnights: {len(lab):,} of {len(uids):,}")
print(lab[["S_ord", "S_emg", "P_early", "P_late", "delta", "delta_placebo"]].median().round(3).to_string())
# the 95% line of the placebo, separately in five bands of how much was seen (fewer records, noisier rise)
edges = np.quantile(lab.seen_min_placebo, [0, .2, .4, .6, .8, 1])
band = lambda v: np.clip(np.searchsorted(edges, v, side="right") - 1, 0, 4)
line = pd.Series(lab.delta_placebo.to_numpy()).groupby(band(lab.seen_min_placebo.to_numpy())).quantile(0.95)
lab["broke"] = lab.delta.to_numpy() > line.reindex(band(lab.seen_min.to_numpy())).to_numpy()
lab["placebo_fires"] = lab.delta_placebo.to_numpy() > line.reindex(band(lab.seen_min_placebo.to_numpy())).to_numpy()
print("95% placebo line by band:", line.round(3).to_dict())
print(f"broke: {lab.broke.mean():.3f} | the same test between two ordinary fortnights fires for {lab.placebo_fires.mean():.3f}")
print(lab.groupby("kind").broke.mean().reindex(ORDER).round(3).to_string())
print("broke by band of seen half hours:", lab.groupby(band(lab.seen_min.to_numpy())).broke.mean().round(3).to_dict())

fig, ax = plt.subplots(figsize=(9.5, 4))
bins = np.linspace(-0.6, 0.9, 76)
ax.hist(lab.delta_placebo, bins=bins, density=True, histtype="stepfilled", color=C[0], alpha=0.35, label="two ordinary fortnights (placebo)")
ax.hist(lab.delta, bins=bins, density=True, histtype="step", lw=2, color=C[1], label="emergency fortnight against the ordinary one")
ax.axvline(line.median(), color="#0b0b0b", lw=1.2, ls="--"); ax.text(line.median() + 0.01, ax.get_ylim()[1] * 0.9, "95% line of the placebo", fontsize=9)
ax.set(xlabel="rise in surprise per seen half hour (nats)", ylabel="density", title=f"Whose habit model broke: {lab.broke.mean():.0%} against 5% by chance")
ax.legend(frameon=False, fontsize=9, loc="center right")
save(fig, "f38_who_broke_label")

In [ ]:
# an alarm per person: add up each day's evidence (CUSUM), set so it fires for only 5% of people on ordinary days 15-44
lab_e = lab[lab.split == "eval"]
Le = L.reindex(lab_e.index).fillna(0.0)               # a day with too few records adds no evidence
rows_k = []
for k_ in [0.0, 0.1, 0.2, 0.3]:
    h = np.percentile(cusum(Le.loc[:, 15:44], k_).max(axis=1), 95)
    al, on = alarm_onset(cusum(Le.loc[:, 45:74], k_), h)
    rows_k.append((k_, h, al[lab_e.broke].notna().mean(), al[~lab_e.broke].notna().mean(), al.median(), on[lab_e.broke].between(57, 63).mean()))
kt = pd.DataFrame(rows_k, columns=["k", "h", "alarm_broke", "alarm_held", "median_alarm_day", "onset_57_63_broke"]).set_index("k")
print(kt.round(3).to_string())
K_ = 0.2
H_ = np.percentile(cusum(Le.loc[:, 15:44], K_).max(axis=1), 95)
C1 = cusum(Le.loc[:, 45:74], K_); alarm, onset = alarm_onset(C1, H_)
lab.loc[lab_e.index, "alarm_day"] = alarm; lab.loc[lab_e.index, "onset_day"] = onset
print(f"k = {K_}, h = {H_:.2f}: alarm for {alarm.notna().mean():.3f} of evaluation people | broke {alarm[lab_e.broke].notna().mean():.3f}, held {alarm[~lab_e.broke].notna().mean():.3f}")
print("alarm by kind:", alarm.notna().groupby(lab_e.kind).mean().reindex(ORDER).round(3).to_dict())
print("alarm days:", pd.cut(alarm.dropna(), [44, 52, 59, 62, 66, 74]).value_counts(sort=False).to_dict())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1.5, 1]})
ex_b = lab_e[lab_e.broke & alarm.notna()].sample(3, random_state=SEED).index; ex_h = lab_e[~lab_e.broke].sample(3, random_state=SEED).index
Cfull = cusum(Le.loc[:, 30:74], K_)
for j, u in enumerate(ex_b): axes[0].plot(Cfull.columns, Cfull.loc[u], color=C[1], lw=1.8, alpha=0.9, label="routine broke" if j == 0 else None)
for j, u in enumerate(ex_h): axes[0].plot(Cfull.columns, Cfull.loc[u], color=C[0], lw=1.8, alpha=0.9, label="routine held" if j == 0 else None)
axes[0].axhline(H_, color="#0b0b0b", ls="--", lw=1); axes[0].axvspan(59.5, 74.5, color=C[1], alpha=0.07, lw=0)
axes[0].set(xlabel="day", ylabel="summed evidence (CUSUM)", title="Six people, evidence added day by day"); axes[0].legend(frameon=False, fontsize=9)
axes[1].hist(alarm[~lab_e.broke].dropna(), bins=np.arange(45, 76), histtype="stepfilled", color=C[0], alpha=0.35, label="routine held")
axes[1].hist(alarm[lab_e.broke].dropna(), bins=np.arange(45, 76), histtype="step", lw=2, color=C[1], label="routine broke")
axes[1].axvline(59.5, color="#52514e", lw=1, ls="--"); axes[1].set(xlabel="day of the alarm", ylabel="people", title="When the alarm fires"); axes[1].legend(frameon=False, fontsize=9)
save(fig, "f39_alarm")

In [ ]:
# trees: who breaks, from days 0-44 only. Features: the 15 Part 2 traits and 3 observability traits (mean of B1-B3), how well the
# person's habits from 0-29 described 30-44, the kind of mover, and the places: daytime place and home
import joblib
sys.path.insert(0, str(CODE)); from profiles import BEHAVIOUR, OBSERVABILITY
prof = pd.read_parquet(PROC / "person_blocks.parquet")
ok = prof[prof.eligible_block & (prof.block <= 2)]
ref = ok.groupby("uid")[BEHAVIOUR + OBSERVABILITY].mean()
fz = joblib.load(PROC / "frozen_part2.joblib")
fit = an[an.d <= 44].copy(); fit["workday"] = days.workday.reindex(fit.d).to_numpy()
cnt = fit.groupby(["x", "y", "workday", "t"]).size().rename("n").reset_index()
busy = cnt.groupby(["x", "y"]).n.sum(); busy = busy[busy >= 300].index
R = cnt.set_index(["x", "y"]).loc[busy].reset_index().pivot_table(index=["x", "y"], columns=["workday", "t"], values="n", fill_value=0)
R = R.reindex(columns=pd.MultiIndex.from_product([[True, False], range(48)]), fill_value=0).to_numpy(dtype="float64")
R[:, :48] /= R[:, :48].sum(axis=1, keepdims=True).clip(min=1); R[:, 48:] /= R[:, 48:].sum(axis=1, keepdims=True).clip(min=1)
rtype = pd.Series(fz["kmeans_rhythm"].predict(R), index=busy).map(fz["rhythm_type_of_label"])     # Part 2 rhythm types, days 0-44 here
cells = pd.read_parquet(PROC / "cell_table.parquet"); cells = cells.reset_index() if "x" not in cells.columns else cells
dens = np.log1p(cells.set_index(["x", "y"]).poi_total)
def at(cell, s): return s.reindex(pd.MultiIndex.from_arrays([cell // 256, cell % 256])).to_numpy()
A = anc.reindex(lab.index)
X = ref.reindex(lab.index).copy()
X["habit_surprise_30_44"] = lab.P_early
X["commute_km"] = np.hypot(A.home // 256 - A.day_anchor // 256, A.home % 256 - A.day_anchor % 256).to_numpy() * 0.5
X["home_places_log"] = at(A.home.to_numpy(), dens); X["daytime_places_log"] = at(A.day_anchor.fillna(-1).astype(int).to_numpy(), dens)
place_type = pd.Series(at(A.day_anchor.fillna(-1).astype(int).to_numpy(), rtype), index=lab.index).fillna("quiet cell")
for k in ORDER: X[f"kind: {k}"] = (lab.kind == k).astype(int)
for v in sorted(place_type.unique()): X[f"daytime place: {v}"] = (place_type == v).astype(int)
y = lab.broke.astype(int)
is_tr, is_te = (lab.split == "dev").to_numpy(), (lab.split == "eval").to_numpy()
fill = X[is_tr].median()
print(f"{X.shape[1]} features | missing values filled with the training median: {int(X.isna().sum().sum())}")
X = X.fillna(fill)
Xtr, Xte, ytr, yte = X[is_tr], X[is_te], y[is_tr], y[is_te]
print(f"training {len(Xtr):,} people ({ytr.mean():.1%} broke) | test {len(Xte):,} ({yte.mean():.1%} broke) | overlap {len(set(Xtr.index) & set(Xte.index))}")
print("daytime place types:", place_type.value_counts().to_dict())
show = ["regularity", "midday_away", "home_share", "rg_km", "records_per_day", "habit_surprise_30_44", "commute_km"]
t14 = ref.reindex(lab.index)[show[:5]].assign(habit_surprise_30_44=lab.P_early, kind=lab.kind, daytime_place=place_type, broke=lab.broke)
t14 = t14[is_tr].sample(6, random_state=SEED).round(3).reset_index()
table_png(t14, "t14_tree_people", "Labelled people for the trees (six of the training set; features from days 0-44, label from 45-74)", 13,
          widths=[0.06, 0.08, 0.09, 0.09, 0.06, 0.11, 0.15, 0.14, 0.11, 0.06])
t15 = X[is_tr].loc[t14.uid, show + [c for c in X.columns if c.startswith("kind:")][:3]].round(3).reset_index()
table_png(t15.rename(columns={"kind: far roamers": "far roamer (0/1)", "kind: mid-range movers": "mid-range (0/1)", "kind: regular commuters": "commuter (0/1)"}),
          "t15_tree_features", "The same people as the trees read them: numbers only, kinds and place types as 0/1 columns (3 of the 12 shown)", 13.5,
          widths=[0.06, 0.08, 0.09, 0.09, 0.06, 0.11, 0.14, 0.085, 0.095, 0.095, 0.095])
t16 = pd.DataFrame({"set": ["training", "test"], "people": [f"{len(Xtr):,}", f"{len(Xte):,}"], "broke": [f"{int(ytr.sum()):,}", f"{int(yte.sum()):,}"],
                    "share broke": [f"{ytr.mean():.3f}", f"{yte.mean():.3f}"], "who": ["development people", "evaluation people"]})
table_png(t16, "t16_tree_split", "Training and test sets for the trees: split by person, as in every part", 7.4, widths=[0.15, 0.13, 0.12, 0.17, 0.43])

In [ ]:
# a split worked by hand: is a person with regularity above 0.71 more likely to break? entropy, Gini and information gain
def ent(p): p = np.asarray(p, float); p = p[p > 0]; return float(-(p * np.log2(p)).sum())
def gini(p): p = np.asarray(p, float); return float(1 - (p ** 2).sum())
cut = 0.71
nodes = {"all training people": ytr, f"regularity <= {cut}": ytr[Xtr.regularity <= cut], f"regularity > {cut}": ytr[Xtr.regularity > cut]}
hand = pd.DataFrame({nm: {"people": len(v), "broke": int(v.sum()), "held": int(len(v) - v.sum()), "share broke": round(v.mean(), 3),
                          "entropy (bits)": round(ent([v.mean(), 1 - v.mean()]), 4), "Gini": round(gini([v.mean(), 1 - v.mean()]), 4)} for nm, v in nodes.items()}).T
print(hand.to_string())
n0, nl, nr = (len(v) for v in nodes.values()); vals = list(nodes.values())
ig = ent([vals[0].mean(), 1 - vals[0].mean()]) - (nl / n0) * ent([vals[1].mean(), 1 - vals[1].mean()]) - (nr / n0) * ent([vals[2].mean(), 1 - vals[2].mean()])
gd = gini([vals[0].mean(), 1 - vals[0].mean()]) - (nl / n0) * gini([vals[1].mean(), 1 - vals[1].mean()]) - (nr / n0) * gini([vals[2].mean(), 1 - vals[2].mean()])
print(f"information gain of the split: {ig:.4f} bits | drop in Gini: {gd:.4f}")

In [ ]:
# three trees: a rule you can say out loud (depth 2), a tree tuned by 5-fold cross-validation, and a tree grown until every leaf is pure.
# 'broke' is the smaller class (about one in five), so class weights make each tree look for it instead of answering 'held'
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
t1 = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=SEED).fit(Xtr, ytr)
gs = GridSearchCV(DecisionTreeClassifier(class_weight="balanced", random_state=SEED),
                  {"criterion": ["gini", "entropy"], "max_depth": [2, 3, 4, 5, 6, 8, 10], "min_samples_leaf": [10, 25, 50, 100, 200]},
                  scoring="roc_auc", cv=cv, n_jobs=-1).fit(Xtr, ytr)
t2 = gs.best_estimator_
t3 = DecisionTreeClassifier(class_weight="balanced", random_state=SEED).fit(Xtr, ytr)
print("cross-validated choice:", gs.best_params_, f"AUC {gs.best_score_:.3f}")
cvr = pd.DataFrame(gs.cv_results_); print(cvr.groupby("param_criterion").mean_test_score.max().round(4).to_dict())
TREES = {"tree 1, depth 2": t1, "tree 2, tuned": t2, "tree 3, grown fully": t3}
tsum = pd.DataFrame({nm: {"leaves": t.get_n_leaves(), "depth": t.get_depth(), "train accuracy": accuracy_score(ytr, t.predict(Xtr)),
                          "test accuracy": accuracy_score(yte, t.predict(Xte)), "test balanced accuracy": balanced_accuracy_score(yte, t.predict(Xte)),
                          "test AUC": roc_auc_score(yte, t.predict_proba(Xte)[:, 1])} for nm, t in TREES.items()}).T
print(tsum.round(3).to_string()); print(f"always answering 'held' would score {1 - yte.mean():.3f} accuracy and 0.5 balanced accuracy")
print(export_text(t1, feature_names=list(X.columns), decimals=2))
print("tuned tree, top features:", pd.Series(t2.feature_importances_, index=X.columns).sort_values(ascending=False).head(8).round(3).to_dict())
fig, axes = plt.subplots(1, 3, figsize=(13, 3.9))
for ax, (nm, t) in zip(axes, TREES.items()):
    cm = confusion_matrix(yte, t.predict(Xte))
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max() * 1.3); ax.grid(False)
    for (r, c_), v in np.ndenumerate(cm): ax.text(c_, r, f"{v:,}\n{v / cm[r].sum():.0%} of row", ha="center", va="center", fontsize=9)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["held", "broke"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["held", "broke"])
    ax.set_xlabel("predicted"); ax.set_title(f"{nm}\naccuracy {accuracy_score(yte, t.predict(Xte)):.1%}, balanced {balanced_accuracy_score(yte, t.predict(Xte)):.1%}", fontsize=10)
axes[0].set_ylabel("actual")
save(fig, "f40_tree_confusion")
fig, ax = plt.subplots(figsize=(11, 4.6)); ax.grid(False)
plot_tree(t1, feature_names=list(X.columns), class_names=["held", "broke"], filled=True, rounded=True, impurity=True, proportion=True, fontsize=9, ax=ax)
ax.set_title("Tree 1: a rule you can say out loud (depth 2)"); save(fig, "f41_tree_depth2")
fig, ax = plt.subplots(figsize=(24, 8)); ax.grid(False)
plot_tree(t2, feature_names=list(X.columns), class_names=["held", "broke"], filled=True, rounded=True, impurity=False, proportion=True, fontsize=9, ax=ax, max_depth=3)
ax.set_title(f"Tree 2: tuned by cross-validation ({t2.get_n_leaves()} leaves; the top four levels shown)"); save(fig, "f42_tree_tuned")

In [ ]:
# why there is never one tree: the pruning path of the fully grown tree, and trees that differ only in their random seed or sample
path = DecisionTreeClassifier(class_weight="balanced", random_state=SEED).cost_complexity_pruning_path(Xtr, ytr)
alphas = np.unique(np.r_[0, np.quantile(path.ccp_alphas, np.linspace(0, 0.995, 40))])
pr_rows = []
for a in alphas:
    t = DecisionTreeClassifier(class_weight="balanced", ccp_alpha=a, random_state=SEED).fit(Xtr, ytr)
    pr_rows.append((a, t.get_n_leaves(), balanced_accuracy_score(ytr, t.predict(Xtr)), balanced_accuracy_score(yte, t.predict(Xte))))
pp = pd.DataFrame(pr_rows, columns=["ccp_alpha", "leaves", "train", "test"]).drop_duplicates("leaves")
print(pp.iloc[::max(1, len(pp) // 12)].round(4).to_string())
roots = []
for b in range(200):
    r = np.random.default_rng(SEED + b).choice(len(Xtr), len(Xtr), replace=True)
    t = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=b).fit(Xtr.iloc[r], ytr.iloc[r])
    tr_ = t.tree_; sig = " | ".join(f"{X.columns[tr_.feature[n]]} {tr_.threshold[n]:.2f}" for n in range(tr_.node_count) if tr_.feature[n] >= 0)
    roots.append((X.columns[tr_.feature[0]], round(tr_.threshold[0], 2), sig, balanced_accuracy_score(yte, t.predict(Xte))))
roots = pd.DataFrame(roots, columns=["first split", "threshold", "tree", "test_balanced"])
print("depth-2 trees on 200 bootstrap samples, first split:", roots["first split"].value_counts().head(6).to_dict(),
      f"| first threshold {roots.threshold.min()} to {roots.threshold.max()} | distinct trees: {roots.tree.nunique()}")
second = roots.tree.str.split(" [|] ").map(lambda v: tuple(sorted(x.rsplit(" ", 1)[0] for x in v[1:])))
print("second-level questions:", second.value_counts().head(5).to_dict())
print(f"their test balanced accuracy: {roots.test_balanced.min():.3f} to {roots.test_balanced.max():.3f}")
fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.4), gridspec_kw={"width_ratios": [1.1, 1], "wspace": 0.38})
ax = axes[0]
ax.plot(pp.leaves, pp.train * 100, color=C[0], lw=2, label="training people"); ax.plot(pp.leaves, pp.test * 100, color=C[1], lw=2, label="test people")
for nm, t in TREES.items():
    ax.axvline(t.get_n_leaves(), color="#c3c2b7", lw=1, ls="--")
    ax.text(t.get_n_leaves(), 0.5, nm.split(", ")[1], transform=ax.get_xaxis_transform(), rotation=90, va="center", ha="right", fontsize=8.5, color="#52514e")
ax.set(xscale="log", xlabel="leaves (fewer = pruned harder)", ylabel="balanced accuracy (%)", title="Pruning: more leaves fit training people, not test people")
ax.legend(frameon=False, fontsize=9, loc="lower left")
ax = axes[1]
vc = second.value_counts().head(5)[::-1]
short = lambda f: f.replace("_km", "").replace("_", " ")
ax.barh([" + ".join(short(f) for f in k) for k in vc.index], vc.values, color=C[2]); ax.yaxis.grid(False)
ax.set(xlabel="of 200 trees", title="200 resampled depth-2 trees: their second questions")
n_reg = int((roots["first split"] == "regularity").sum())
ax.text(0.98, 0.04, f"first question: regularity in {n_reg} of 200, cut at {roots.threshold.min():.2f} to {roots.threshold.max():.2f}\n"
        f"{roots.tree.nunique()} distinct trees, test balanced accuracy {roots.test_balanced.min():.1%} to {roots.test_balanced.max():.1%}",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=8.5, color="#52514e", bbox=dict(fc="#fcfcfb", ec="#e1e0d9"))
save(fig, "f43_pruning_and_many_trees")

In [ ]:
# who breaks, without a model: share broke by regularity, by kind and by the kind of daytime place
d_ = X.assign(broke=y, kind=lab.kind, place=place_type)
by_reg = d_.groupby(pd.qcut(d_.regularity, 5, duplicates="drop"), observed=True).broke.agg(["mean", "size"])
by_kind = d_.groupby("kind").broke.agg(["mean", "size"]).reindex(ORDER)
by_place = d_.groupby("place").broke.agg(["mean", "size"]).sort_values("mean")
print(by_reg.round(3).to_string()); print(by_kind.round(3).to_string()); print(by_place.round(3).to_string())
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True, gridspec_kw={"width_ratios": [0.85, 1.3, 1.2]})
panels = [(by_reg, ["#7d838c"] * len(by_reg), ["least\nregular", "", "", "", "most\nregular"], "By regularity (fifths)"),
          (by_kind, [COL[k] for k in ORDER], [k.replace(" ", "\n", 1) for k in ORDER], "By kind of mover"),
          (by_place, ["#7d838c"] * len(by_place), [p_.replace(" ", "\n", 1) for p_ in by_place.index], "By the rhythm of the daytime place")]
for ax, (tab, cols_, labels, title) in zip(axes, panels):
    xx = np.arange(len(tab)); ax.bar(xx, tab["mean"] * 100, color=cols_, width=0.7)
    for x_, v in zip(xx, tab["mean"]): ax.text(x_, v * 100 + 0.5, f"{v:.0%}", ha="center", fontsize=8.5, color="#52514e")
    ax.set_xticks(xx); ax.set_xticklabels(labels, fontsize=8.5); ax.set_title(title)
    ax.axhline(y.mean() * 100, color="#52514e", lw=1, ls="--"); ax.xaxis.grid(False)
axes[0].set_ylabel("% whose routine broke")
save(fig, "f44_who_breaks")

In [ ]:
# outputs for the foundation-model notebook and the site
SAMPLE = ROOT / "data/samples"; SAMPLE.mkdir(parents=True, exist_ok=True)
X_out = X.assign(broke=y, split=lab.split, kind=lab.kind)
X_out.to_parquet(PROC / "who_breaks_table.parquet")
lab.to_parquet(PROC / "person_breaks.parquet")
shares = pd.DataFrame({"uid": np.repeat(uids, 75), "d": np.tile(np.arange(75), len(uids)),
                       "seen": (S[:, :, SLOTS] > 0).sum(axis=2).ravel(), "home": (S[:, :, SLOTS] == HOME).sum(axis=2).ravel(),
                       "place": (S[:, :, SLOTS] == PLACE).sum(axis=2).ravel()})
shares.to_parquet(PROC / "person_day_shares.parquet", index=False)
L.to_parquet(PROC / "day_evidence_eval.parquet")
eb_tab.to_csv(PROC / "nb_detectors_eval.csv", index=False); summary.to_csv(PROC / "nb_models_eval.csv")
pd.DataFrame({"y": yte, **{nm: t.predict_proba(Xte)[:, 1] for nm, t in TREES.items()}}).to_parquet(PROC / "tree_test_scores.parquet")
ex_out = rows.sample(500, random_state=SEED)
pd.DataFrame({"uid": ex_out.uid, "day": ex_out.d, "rest_day": ex_out.rest, "label": ex_out.y,
              "half_hours": ["".join(LETTER[S[i, d, SLOTS]]) for i, d in zip(ex_out.i, ex_out.d)]}).to_csv(SAMPLE / "nb_days_sample.csv", index=False)
X_out.sample(500, random_state=SEED).round(4).to_csv(SAMPLE / "who_breaks_sample.csv")
log = {"people": len(uids), "nb_days": {"train": len(train), "val": len(val), "test": len(test)}, "best_alpha": best_alpha.to_dict(),
       "nb_test": summary.round(4).to_dict(), "eb_prior_strength_person": m_p, "eb_prior_strength_kind": {k: list(v) for k, v in mk.items()},
       "broke_share": round(float(lab.broke.mean()), 4), "placebo_share": round(float(lab.placebo_fires.mean()), 4),
       "broke_by_kind": lab.groupby("kind").broke.mean().round(4).to_dict(), "cusum": {"k": K_, "h": round(float(H_), 3)},
       "trees": tsum.round(4).to_dict(), "tuned_tree": gs.best_params_}
(ROOT / "logs/part3_log.json").write_text(json.dumps(log, indent=2, default=str)); print(json.dumps(log, indent=2, default=str)[:3000])
for p_ in ["who_breaks_table.parquet", "person_breaks.parquet", "person_day_shares.parquet", "day_evidence_eval.parquet"]:
    print(f"{p_:28s} {(PROC / p_).stat().st_size / 1e6:6.1f} MB")

In [ ]:
# one archive with what this notebook produced, to download from the file browser. Press Ctrl+S first, then run this cell again,
# so the archive holds the notebook with its outputs
import zipfile
out = ROOT / "part3_outputs.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(FIG.glob("f3[3-9]_*.png")) + sorted(FIG.glob("f4[0-4]_*.png")) + sorted(FIG.glob("t1[1-6]_*.png")):
        z.write(p, f"figures/{p.name}")
    for p in [ROOT / "code/breaks.py", ROOT / "logs/part3_log.json", SAMPLE / "nb_days_sample.csv", SAMPLE / "who_breaks_sample.csv",
              ROOT / "notebooks/04_when_and_who.ipynb"]:
        if p.exists(): z.write(p, str(p.relative_to(ROOT)))
        else: print("missing:", p)
print(out, f"{out.stat().st_size / 1e6:.1f} MB")